# Explore: YOLOv8 + KServe inference with supervision visualization

This notebook sends a test image to the KServe `InferenceService` and draws
predicted bounding boxes using the [supervision](https://supervision.roboflow.com/) library.

**Run this notebook from the `cv-lab-notebook` JupyterLab instance** inside the
`cv-lab` Kubeflow profile. The notebook pod sits in the `cv-lab` namespace so
Istio mTLS identities allow it to reach the predictor service directly.

In [ ]:
# Install dependencies (idempotent; skip if already installed)
%pip install -q supervision pillow requests

In [ ]:
import base64
import io

import numpy as np
import requests
import supervision as sv
from PIL import Image

## Configuration

In [ ]:
# KServe predictor service (reachable in-cluster from the cv-lab namespace)
INFER_URL = "http://yolov8-coco128-predictor.cv-lab/v1/models/yolov8-coco128:predict"

# Test image — replace with any accessible URL or local path
TEST_IMAGE_URL = "https://ultralytics.com/images/zidane.jpg"

## Load test image

In [ ]:
resp = requests.get(TEST_IMAGE_URL, timeout=30)
resp.raise_for_status()
image = Image.open(io.BytesIO(resp.content)).convert("RGB")
print(f"Image size: {image.size}")
image

## Send to KServe for inference

In [ ]:
def image_to_b64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode()


payload = {"instances": [{"image": {"b64": image_to_b64(image)}}]}
r = requests.post(INFER_URL, json=payload, timeout=60)
r.raise_for_status()
result = r.json()

preds = result["predictions"][0]
print(f"Detections: {len(preds['boxes'])}")
for box, score, cls in zip(preds["boxes"], preds["scores"], preds["class_names"]):
    print(f"  {cls}: {score:.2f}  {[round(v) for v in box]}")

## Visualize with supervision

In [ ]:
img_np = np.array(image)

boxes = np.array(preds["boxes"], dtype=np.float32)          # (N, 4) xyxy
scores = np.array(preds["scores"], dtype=np.float32)         # (N,)
class_ids = np.array(preds["class_ids"], dtype=int)          # (N,)
class_names = preds["class_names"]                            # list[str]

# Build supervision Detections object
detections = sv.Detections(
    xyxy=boxes,
    confidence=scores,
    class_id=class_ids,
)

# Labels include class name + confidence
labels = [
    f"{cls} {score:.2f}"
    for cls, score in zip(class_names, scores.tolist())
]

# Annotate
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

# Display
Image.fromarray(annotated)

## Run on a custom image

Replace `TEST_IMAGE_URL` at the top of this notebook with a different URL or load
a local file with `Image.open("/path/to/image.jpg").convert("RGB")` and re-run
the **Send** and **Visualize** cells.